## Team Members

| # | Name | GitHub |
|---|------|--------|
| 1 | Rohima Ahmed Ibrahim | [rohimaahmed101-sys](https://github.com/rohimaahmed101-sys) |
| 2 | Shaimaa Kotit | [shaimaakotit-29](https://github.com/shaimaakotit-29) |
| 3 | Doha Mahmoud | [dohaalaraby-lang](https://github.com/dohaalaraby-lang) |

**Dataset:** [Skin Cancer MNIST: HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000)

# AIE231 Course Project: 01 — Exploratory Data Analysis and data split (Phase 2)

## What this notebook does
1. Builds the **lesion-level, stratified 70 / 15 / 15 split** with `src/data.py` and saves it to `data/splits/split_v1.csv`.
2. Runs every **leakage check** (no lesion, group or identical image in two splits).
3. Explores the data: class balance per split, photos per lesion, age / sex / body site per class, how labels were confirmed, and image brightness and colour.
4. Checks the **input pipeline**: one batch from `make_dataset`, value ranges per model family, and a preview of the training augmentation.
5. Saves `experiments/eda_summary.json` with every number shown here.

All reusable code lives in `src/data.py`; this notebook only **imports** it (spec 6.1: notebooks import from `src/`).

| Checklist item | Spec rule | Where |
|---|---|---|
| E1 | Separate train / val / test; test used once at the end | Section 2 |
| E2 | Stratified; samples from one source stay in one split | Sections 2–3 |
| E3 | No leakage | Section 2 |
| E4 | Duplicates handled, method justified | Section 2 |
| E5 | Scaling and encoding | Section 8 |
| E7 | Imbalance measured and addressed | Sections 3 and 7 |
| E8 | At least two augmentations, training data only | Section 8 |
| P3 / P4 | EDA with findings linked to decisions | Sections 3–9 |

## Test-set discipline
Image-content statistics (Section 6) and the augmentation preview use **training images only**. The test split is counted (class balance) but never looked at, so no decision is influenced by it.

## How to run
1. Push `src/data.py` and `configs/base.yaml` to the branch `feature/phase2-data-pipeline` first (this notebook clones that branch).
2. **Runtime → Change runtime type → T4 GPU** (Section 8 uses TensorFlow).
3. `KAGGLE_API_TOKEN` must be in Colab Secrets (same as Phase 1).
4. **Runtime → Run all** (about 5–10 minutes; the MD5 duplicate check takes 1–3 of them).
5. Download `data/splits/split_v1.csv` and `experiments/eda_summary.json` and send the outputs for review. The split file is then committed to the same branch.

## 0. Setup: token, repository, configuration

In [ ]:
# 0a. Kaggle token from Colab Secrets (never written in the notebook)
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
print("Kaggle token loaded from Colab Secrets.")

In [ ]:
# 0b. Clone the project branch and install the pinned packages
REPO_URL = "https://github.com/rohimaahmed101-sys/ham10000-skin-lesion-classification.git"
BRANCH = "feature/phase2-data-pipeline"
REPO_DIR = "/content/ham10000-skin-lesion-classification"

if not os.path.exists(REPO_DIR):
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
# 0c. Imports and configuration (everything reusable comes from src/)
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, str(Path.cwd()))
from src.config import load_config, set_seed
from src import data as D

cfg = load_config("configs/base.yaml")
set_seed(cfg["seed"])
CLASSES = cfg["data"]["classes"]
DX_NAMES = {"akiec": "Actinic keratoses / IEC", "bcc": "Basal cell carcinoma", "bkl": "Benign keratosis-like",
            "df": "Dermatofibroma", "mel": "Melanoma", "nv": "Melanocytic nevi", "vasc": "Vascular lesions"}
SPLIT_COLORS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}   # validated categorical palette
INK, QUIET, GRID = "#0b0b0b", "#52514e", "#e5e4df"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": QUIET, "xtick.color": QUIET, "ytick.color": QUIET,
                     "axes.titleweight": "bold", "axes.titlesize": 12, "font.size": 10})
EDA = {}   # every number below is stored here and saved to eda_summary.json
print("Config loaded. Seed:", cfg["seed"], "| split file:", cfg["data"]["split_file"])

## 1–2. Build the split and run the leakage checks (E1, E2, E3, E4)
`D.prepare_split(cfg)`:
1. loads the metadata and maps each `image_id` to **one** file;
2. finds byte-identical images (MD5) and merges their lesions into one **group** (handles the 2 duplicate pairs found in Phase 1 without deleting data);
3. splits **groups** 70 / 15 / 15, stratified by class, with seed 42, and saves `data/splits/split_v1.csv` (if the file already exists in the branch it is loaded, not rebuilt);
4. runs `check_no_leakage`: every image once, no lesion or group in two splits, identical images in one split, every class in every split. Any failure stops the notebook.

In [ ]:
meta, split_df, identical, report = D.prepare_split(cfg)
EDA["identical_groups"] = identical
EDA["leakage_report"] = report
print("Identical image groups merged into one group:", identical)
print("Leakage checks:", report)
print("Split file:", D.split_path(cfg))

## 3. Class balance in each split (E1, E2, E7)
Groups are split, not images, so the image shares are close to, but not exactly, 70 / 15 / 15. Stratification should give every split a similar class mix.

In [ ]:
counts = D.split_summary(split_df, CLASSES)
shares = D.split_shares(split_df)
props = D.class_proportions(split_df, CLASSES)
EDA["split_counts"] = counts.to_dict()
EDA["split_shares_pct"] = shares
EDA["class_pct_per_split"] = props.to_dict()
EDA["max_class_pct_gap_between_splits"] = float((props.max(axis=1) - props.min(axis=1)).max())

print("Images per class and split:")
print(counts.to_string())
print("\nShare of images per split (%):", shares)
print("\nClass percentage inside each split:")
print(props.to_string())
print("\nLargest gap of one class's percentage between splits (percentage points):",
      round(EDA["max_class_pct_gap_between_splits"], 2))

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(CLASSES))
w = 0.26
for k, name in enumerate(D.SPLITS):
    ax.bar(x + (k - 1) * w, props[name].values, width=w - 0.02, color=SPLIT_COLORS[name], label=name)
ax.set_xticks(x, CLASSES)
ax.set_ylabel("% of images in the split")
ax.set_title("Class mix per split (stratified by lesion group)")
ax.grid(axis="y", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 4. Photos per lesion, by class
How many images each class gets from repeated photos of the same lesion. Classes with many repeats have fewer **independent** examples than their image count suggests.

In [ ]:
per_lesion = meta.groupby(["dx", "lesion_id"]).size().rename("n").reset_index()
lesion_stats = per_lesion.groupby("dx").agg(images=("n", "sum"), lesions=("lesion_id", "count"),
                                            mean_photos_per_lesion=("n", "mean"),
                                            lesions_with_2plus=("n", lambda s: int((s > 1).sum())))
lesion_stats["pct_lesions_with_2plus"] = (lesion_stats["lesions_with_2plus"] / lesion_stats["lesions"] * 100).round(1)
lesion_stats["mean_photos_per_lesion"] = lesion_stats["mean_photos_per_lesion"].round(2)
lesion_stats = lesion_stats.reindex(CLASSES)
EDA["lesion_stats"] = lesion_stats.to_dict()
print(lesion_stats.to_string())

## 5. Patient metadata and label confirmation by class
- **Age, sex, body site:** are some classes concentrated in certain groups? (Useful context for the error analysis; the models use images only.)
- **`dx_type`** — how each label was confirmed: `histo` (histopathology, the most reliable), `follow_up`, `consensus` (expert agreement), `confocal`. Classes with many consensus labels may contain more label noise.

In [ ]:
# Age
age = meta.groupby("dx")["age"].describe()[["count", "mean", "50%", "min", "max"]].reindex(CLASSES).round(1)
age = age.rename(columns={"50%": "median"})
EDA["age_by_class"] = age.to_dict()
print("Age by class (NaN excluded):")
print(age.to_string())

fig, ax = plt.subplots(figsize=(7.5, 3.6))
data = [meta.loc[meta["dx"] == c, "age"].dropna().values for c in CLASSES]
ax.boxplot(data, vert=False, tick_labels=CLASSES, widths=0.55, patch_artist=True,
           boxprops=dict(facecolor="#cde2fb", edgecolor=QUIET), medianprops=dict(color="#2a78d6", linewidth=2),
           whiskerprops=dict(color=QUIET), capprops=dict(color=QUIET), flierprops=dict(markersize=3, markeredgecolor=QUIET))
ax.set_xlabel("Age (years)"); ax.set_title("Age distribution by class")
ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

In [ ]:
# Sex and body site (percent within each class)
sex = (pd.crosstab(meta["dx"], meta["sex"], normalize="index") * 100).reindex(CLASSES).round(1)
EDA["sex_pct_by_class"] = sex.to_dict()
print("Sex (% within class):")
print(sex.to_string())

top_sites = meta["localization"].value_counts().index[:8]
site = (pd.crosstab(meta["dx"], meta["localization"], normalize="index") * 100)[top_sites].reindex(CLASSES).round(1)
EDA["top_sites_pct_by_class"] = site.to_dict()
print("\nBody site, 8 most common overall (% within class):")
print(site.to_string())

In [ ]:
# How labels were confirmed (percent within each class)
dxt = (pd.crosstab(meta["dx"], meta["dx_type"], normalize="index") * 100).reindex(CLASSES).round(1)
EDA["dx_type_pct_by_class"] = dxt.to_dict()
print("dx_type (% within class):")
print(dxt.to_string())

## 6. Image appearance — training split only
Mean brightness and mean R, G, B of up to 150 training images per class (read at 64 × 64 for speed, seed 42). Large differences between classes could become a shortcut the model learns instead of the lesion itself.

In [ ]:
rng_state = cfg["seed"]
train_df = split_df[split_df["split"] == "train"]
rows = []
for c in CLASSES:
    pool = train_df[train_df["dx"] == c]
    for p in pool.sample(n=min(150, len(pool)), random_state=rng_state)["path"]:
        with Image.open(p) as img:
            a = np.asarray(img.convert("RGB").resize((64, 64)), dtype=np.float32)
        rows.append({"dx": c, "brightness": a.mean(), "R": a[..., 0].mean(), "G": a[..., 1].mean(), "B": a[..., 2].mean()})
appearance = pd.DataFrame(rows)
app_stats = appearance.groupby("dx")[["brightness", "R", "G", "B"]].median().reindex(CLASSES).astype(float).round(1)
EDA["appearance_median_train"] = app_stats.to_dict()
print("Median over sampled TRAINING images (0-255):")
print(app_stats.to_string())

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.boxplot([appearance.loc[appearance["dx"] == c, "brightness"] for c in CLASSES], vert=False, tick_labels=CLASSES,
           widths=0.55, patch_artist=True, boxprops=dict(facecolor="#cde2fb", edgecolor=QUIET),
           medianprops=dict(color="#2a78d6", linewidth=2), whiskerprops=dict(color=QUIET), capprops=dict(color=QUIET),
           flierprops=dict(markersize=3, markeredgecolor=QUIET))
ax.set_xlabel("Mean pixel brightness (0-255)"); ax.set_title("Image brightness by class (training images)")
ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

## 7. Class weights (E7)
Balanced weights from the **training split only**: `n_samples / (n_classes × count)`. Rare classes get a larger weight in the loss, so missing them costs more. The effect on per-class recall is measured later as an ablation (with vs. without class weights).

In [ ]:
weights = D.class_weights(split_df, len(CLASSES))
EDA["class_weights_train"] = {CLASSES[i]: round(w, 4) for i, w in weights.items()}
print(pd.Series(EDA["class_weights_train"], name="weight").to_string())

## 8. Input pipeline check and augmentation preview (E5, E8)
- Builds a small training dataset (with augmentation) and a validation dataset (without) through `D.make_dataset`.
- Prints the batch shape, label encoding and the pixel range after each model family's scaling.
- Shows one training image and 7 augmented versions (flip, rotation, zoom, brightness, contrast).

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

small_train = train_df.groupby("dx", group_keys=False).head(10)
small_val = split_df[split_df["split"] == "val"].groupby("dx", group_keys=False).head(5)
pipeline = {}
for fam in ["cnn", "resnet50", "vgg16"]:
    x, y = next(iter(D.make_dataset(small_train, cfg, fam, training=True, batch_size=16)))
    xv, yv = next(iter(D.make_dataset(small_val, cfg, fam, training=False, batch_size=16)))
    pipeline[fam] = {"batch_shape": list(x.shape), "label_shape": list(y.shape), "label_row_sums": float(tf.reduce_mean(tf.reduce_sum(y, 1))),
                     "train_min": round(float(tf.reduce_min(x)), 2), "train_max": round(float(tf.reduce_max(x)), 2),
                     "val_min": round(float(tf.reduce_min(xv)), 2), "val_max": round(float(tf.reduce_max(xv)), 2)}
EDA["pipeline_check"] = pipeline
print(pd.DataFrame(pipeline).T.to_string())

In [ ]:
# Augmentation preview: one training image, original + 7 augmented versions
aug = D.build_augmenter(cfg["augmentation"], cfg["seed"])
sample_path = train_df.sample(1, random_state=cfg["seed"])["path"].iloc[0]
img = tf.cast(D.read_and_resize(sample_path, tuple(cfg["data"]["image_size"])), tf.float32)
batch = tf.repeat(img[None], 7, axis=0)
augmented = aug(batch, training=True)

fig, axes = plt.subplots(2, 4, figsize=(9, 4.8))
axes = axes.ravel()
axes[0].imshow(img.numpy().astype("uint8")); axes[0].set_title("original", fontsize=9)
for i in range(7):
    axes[i + 1].imshow(np.clip(augmented[i].numpy(), 0, 255).astype("uint8")); axes[i + 1].set_title(f"augmented {i + 1}", fontsize=9)
for a in axes: a.axis("off")
fig.suptitle("Training augmentation: flip, rotation, zoom, brightness, contrast", fontweight="bold")
plt.tight_layout(); plt.show()

## 9. Save the summary
`experiments/eda_summary.json` holds every number above. The findings → decisions table for the report is written from these outputs after review.

In [ ]:
def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    return o

Path("experiments/eda_summary.json").write_text(json.dumps(to_jsonable(EDA), indent=2))
print("Saved:", Path("experiments/eda_summary.json").resolve())
print("Split file to commit:", D.split_path(cfg))

In [ ]:
# Download both files to your computer
from google.colab import files
files.download(str(D.split_path(cfg)))
files.download("experiments/eda_summary.json")